# 00b — Train / Validation Split

**Phase 0, Step 0.2** — Hold out 10% of training S1 entities for validation.

Stratification: by `country` × `match_count_bracket` (0, 1, 2, 3-5, 6+).

Outputs saved to `dataset/splits/`:
- `val_s1_ids.txt`         — validation S1 entity IDs (one per line)
- `train_s1_ids.txt`       — training S1 entity IDs
- `val_ground_truth.tsv`   — GT rows for validation entities
- `train_ground_truth.tsv` — GT rows for training entities

In [1]:
from __future__ import annotations
import os
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedShuffleSplit

# --- Paths -------------------------------------------------------------------
REPO_ROOT = Path(os.getcwd())
for p in [Path(os.getcwd()), *Path(os.getcwd()).parents]:
    if (p / "dataset").exists():
        REPO_ROOT = p
        break

TRAIN_DIR  = REPO_ROOT / "dataset" / "train"
SPLITS_DIR = REPO_ROOT / "dataset" / "splits"
SPLITS_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED  = 42
VAL_FRACTION = 0.10

print(f"REPO_ROOT  : {REPO_ROOT}")
print(f"SPLITS_DIR : {SPLITS_DIR}")

REPO_ROOT  : c:\Users\arula\Documents\Work\ML_challenge
SPLITS_DIR : c:\Users\arula\Documents\Work\ML_challenge\dataset\splits


## 1. Load Source 1 and Ground Truth

In [2]:
print("Loading train_source1.tsv ...")
s1 = pd.read_csv(
    TRAIN_DIR / "train_source1.tsv",
    sep="\t", dtype=str, low_memory=False,
    usecols=["entity_id", "country"],
)
s1 = s1.rename(columns={"entity_id": "source1_entity_id"})
print(f"  S1 rows: {len(s1):,}")

print("Loading train_ground_truth.tsv ...")
gt = pd.read_csv(
    TRAIN_DIR / "train_ground_truth.tsv",
    sep="\t", dtype=str, low_memory=False,
)
print(f"  GT rows: {len(gt):,}")
print(f"  GT columns: {list(gt.columns)}")

Loading train_source1.tsv ...
  S1 rows: 2,206,821
Loading train_ground_truth.tsv ...
  GT rows: 2,206,821
  GT columns: ['source1_entity_id', 'matched_entity_ids']


## 2. Compute match_count_bracket for stratification

In [3]:
def match_count_bracket(raw):
    if not isinstance(raw, str) or not raw.strip():
        return "0"       # singleton
    n = len(raw.split(","))
    if n == 1:  return "1"
    if n == 2:  return "2"
    if n <= 5:  return "3-5"
    return "6+"

gt["bracket"] = gt["matched_entity_ids"].apply(match_count_bracket)

# Merge country into GT
gt = gt.merge(s1[["source1_entity_id", "country"]], on="source1_entity_id", how="left")

# Stratification key
gt["strat_key"] = gt["country"].fillna("UNKNOWN") + "__" + gt["bracket"]

print("\nStratification key distribution:")
print(gt["strat_key"].value_counts().to_string())


Stratification key distribution:
strat_key
US__3-5       801940
India__3-5    534973
US__2         225285
US__6+        150823
India__2      149927
India__6+     101469
US__0          73896
US__1          71689
India__0       49351
India__1       47468


## 3. Stratified split

In [4]:
X = gt.index.to_numpy().reshape(-1, 1)
y = gt["strat_key"].to_numpy()

sss = StratifiedShuffleSplit(n_splits=1, test_size=VAL_FRACTION, random_state=RANDOM_SEED)
train_idx, val_idx = next(sss.split(X, y))

gt_train = gt.iloc[train_idx].copy()
gt_val   = gt.iloc[val_idx].copy()

print(f"Training set  : {len(gt_train):,} S1 entities")
print(f"Validation set: {len(gt_val):,} S1 entities ({len(gt_val)/len(gt)*100:.1f}%)")

# Verify stratification quality
train_dist = gt_train["strat_key"].value_counts(normalize=True).rename("train")
val_dist   = gt_val["strat_key"].value_counts(normalize=True).rename("val")
strat_check = pd.concat([train_dist, val_dist], axis=1).round(4)
print("\nStratification check (proportions):")
print(strat_check.to_string())

Training set  : 1,986,138 S1 entities
Validation set: 220,683 S1 entities (10.0%)

Stratification check (proportions):
             train     val
strat_key                 
US__3-5     0.3634  0.3634
India__3-5  0.2424  0.2424
US__2       0.1021  0.1021
US__6+      0.0683  0.0683
India__2    0.0679  0.0679
India__6+   0.0460  0.0460
US__0       0.0335  0.0335
US__1       0.0325  0.0325
India__0    0.0224  0.0224
India__1    0.0215  0.0215


## 4. Save split files

In [5]:
out_cols = ["source1_entity_id", "matched_entity_ids"]

# Ground truth subsets
gt_train[out_cols].to_csv(SPLITS_DIR / "train_ground_truth.tsv", sep="\t", index=False)
gt_val[out_cols].to_csv(SPLITS_DIR / "val_ground_truth.tsv", sep="\t", index=False)

# ID lists
(SPLITS_DIR / "train_s1_ids.txt").write_text(
    "\n".join(gt_train["source1_entity_id"].tolist()), encoding="utf-8"
)
(SPLITS_DIR / "val_s1_ids.txt").write_text(
    "\n".join(gt_val["source1_entity_id"].tolist()), encoding="utf-8"
)

# Country composition summary
summary = {
    "train": gt_train["country"].value_counts().to_dict(),
    "val":   gt_val["country"].value_counts().to_dict(),
}
print("\nSaved to", SPLITS_DIR)
print("Split files:")
for f in sorted(SPLITS_DIR.iterdir()):
    print(f"  {f.name}  ({f.stat().st_size:,} bytes)")

print("\nCountry composition:")
for split_name, counts in summary.items():
    print(f"  {split_name}: {counts}")


Saved to c:\Users\arula\Documents\Work\ML_challenge\dataset\splits
Split files:
  train_ground_truth.tsv  (116,297,481 bytes)
  train_s1_ids.txt  (27,585,389 bytes)
  val_ground_truth.tsv  (12,924,962 bytes)
  val_s1_ids.txt  (3,064,935 bytes)

Country composition:
  train: {'US': 1191269, 'India': 794869}
  val: {'US': 132364, 'India': 88319}


## 5. Sanity check: no overlap between train and val

In [6]:
train_ids = set(gt_train["source1_entity_id"])
val_ids   = set(gt_val["source1_entity_id"])
overlap   = train_ids & val_ids

assert len(overlap) == 0, f"OVERLAP DETECTED: {len(overlap)} shared IDs"
assert len(train_ids) + len(val_ids) == len(gt), "Partition size mismatch"

print("Sanity check PASSED:")
print(f"  No overlap between train and val")
print(f"  train + val = {len(train_ids):,} + {len(val_ids):,} = {len(gt):,} total")

Sanity check PASSED:
  No overlap between train and val
  train + val = 1,986,138 + 220,683 = 2,206,821 total
